In [ ]:
!apt-get update -qq
!apt-get install -y -qq \
    python3.12 \
    python3.12-venv \
    python3.12-dev

In [94]:
!apt-get install -y -qq \
    build-essential \
    cmake \
    pkg-config \
    libsndfile1-dev \
    libfftw3-dev \
    libhdf5-dev \
    ffmpeg \
    curl

In [ ]:
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y

In [ ]:
import os

os.environ["PATH"] = "/root/.cargo/bin:" + os.environ["PATH"]

!rustc --version
!cargo --version

In [ ]:
import os
import subprocess

ENV_DIR = "/content/dfenv"
PYTHON = f"{ENV_DIR}/bin/python"
PIP = f"{ENV_DIR}/bin/pip"

if not os.path.exists(PYTHON):
    subprocess.run(
        ["python3.12", "-m", "venv", ENV_DIR],
        check=True
    )

print("DeepFilterNet environment created")
print(subprocess.check_output([PYTHON, "--version"], text=True).strip())

In [78]:
!{PIP} install -q --upgrade pip setuptools wheel

In [ ]:
!{PIP} install -q "numpy==1.26.4"

In [80]:
!{PIP} install -q \
    "torch==2.3.1" \
    "torchaudio==2.3.1"

In [81]:
!{PIP} install -q "maturin==1.4.0"

# DeepFilterNet's runtime dependencies
!{PIP} install -q loguru soundfile librosa

In [ ]:
import os
import subprocess

REPO = "/content/DeepFilterNet"

if not os.path.exists(REPO):
    subprocess.run(
        [
            "git",
            "clone",
            "--depth", "1",
            "https://github.com/Rikorose/DeepFilterNet.git",
            REPO
        ],
        check=True
    )
else:
    print("DeepFilterNet repository already exists.")

print("Repository:", os.path.exists(REPO))
print("pyDF:", os.path.exists(f"{REPO}/pyDF"))

In [ ]:
import subprocess
import os

env = os.environ.copy()

env["PYO3_PYTHON"] = "/usr/bin/python3.12"
env["PYTHON_SYS_EXECUTABLE"] = "/usr/bin/python3.12"

print("Building libDF...")
print("PyO3 Python:", env["PYO3_PYTHON"])


result = subprocess.run(
    [
        "cargo",
        "build",
        "--release",
        "--manifest-path",
        "/content/DeepFilterNet/pyDF/Cargo.toml"
    ],
    cwd="/content/DeepFilterNet",
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    raise RuntimeError(
        f" Cargo build failed: {result.returncode}"
    )

print(" libDF Cargo build successful!")

In [ ]:
import os
import subprocess

PYTHON312 = "/usr/bin/python3.12"
CARGO_TOML = "/content/DeepFilterNet/pyDF/Cargo.toml"

env = os.environ.copy()

# Force PyO3 to Python 3.12
env["PYO3_PYTHON"] = PYTHON312
env["PYTHON_SYS_EXECUTABLE"] = PYTHON312

# Putting Python 3.12 first
env["PATH"] = "/usr/bin:/bin:" + env["PATH"]

# Remove PyO3 variables
for key in [
    "PYO3_CROSS",
    "PYO3_CROSS_LIB_DIR",
    "PYO3_CROSS_PYTHON_VERSION",
    "PYO3_CROSS_PYTHON_IMPLEMENTATION",
    "PYO3_PRINT_CONFIG",
]:
    env.pop(key, None)

print("PYO3_PYTHON:", env["PYO3_PYTHON"])
print("Python:", subprocess.check_output(
    [PYTHON312, "--version"], text=True
).strip())

print("\nBuilding Python extension...")


result = subprocess.run(
    [
        "/content/dfenv/bin/maturin",
        "build",
        "--release",
        "--interpreter",
        PYTHON312,
        "--manifest-path",
        CARGO_TOML
    ],
    cwd="/content/DeepFilterNet",
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    raise RuntimeError(
        f" Maturin failed with exit code {result.returncode}"
    )

print("\nPython extension built successfully!")

In [ ]:
import glob

wheels = glob.glob(
    "/content/DeepFilterNet/target/wheels/*.whl"
)

print("Wheels:")

for wheel in wheels:
    print(wheel)

if not wheels:
    raise RuntimeError(" No wheel was generated.")

LIBDF_WHEEL = wheels[-1]

print("\n Selected:")
print(LIBDF_WHEEL)

In [ ]:
import subprocess

subprocess.run(
    [
        "/content/dfenv/bin/pip",
        "install",
        "--force-reinstall",
        LIBDF_WHEEL
    ],
    check=True
)

print("libDF Python extension installed")

In [87]:
ENHANCE = "/content/DeepFilterNet/DeepFilterNet/df/enhance.py"

In [ ]:
from google.colab import files
import os

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No file uploaded.")

INPUT_FILE = os.path.abspath(next(iter(uploaded)))

print("Uploaded:")
print(INPUT_FILE)

In [ ]:
import subprocess

RAW_WAV = "/content/ptt_raw_48k.wav"

result = subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i", INPUT_FILE,
        "-ar", "48000",
        "-ac", "1",
        "-sample_fmt", "s16",
        RAW_WAV
    ],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(" FFmpeg conversion failed.")

print("Converted successfully:")
print(RAW_WAV)

In [ ]:
import os
import subprocess

OUTPUT_DIR = "/content/enhanced"
os.makedirs(OUTPUT_DIR, exist_ok=True)

env = os.environ.copy()

# DeepFilterNet's `df` package
env["PYTHONPATH"] = (
    "/content/DeepFilterNet/DeepFilterNet"
    + ":"
    + env.get("PYTHONPATH", "")
)

# native Python extension available.
env["PYO3_PYTHON"] = "/usr/bin/python3.12"
env["PYTHON_SYS_EXECUTABLE"] = "/usr/bin/python3.12"

print("PYTHONPATH:")
print(env["PYTHONPATH"])

print("\nRunning DeepFilterNet3...")


result = subprocess.run(
    [
        PYTHON,
        ENHANCE,
        RAW_WAV,
        "--output-dir",
        OUTPUT_DIR,
        "--model-base-dir",
        "DeepFilterNet"
    ],
    cwd="/content/DeepFilterNet",
    env=env,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("\n--- DeepFilterNet log ---")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f" DeepFilterNet3 enhancement failed with exit code {result.returncode}"
    )

print("\n DeepFilterNet3 completed successfully!")

In [ ]:
import glob
import os

enhanced_files = glob.glob("/content/enhanced/*.wav")

print("Enhanced files:")
for f in enhanced_files:
    print(f)

if not enhanced_files:
    raise RuntimeError("Enhanced file not found.")

ENHANCED_WAV = enhanced_files[0]

print("\n Enhanced file:")
print(ENHANCED_WAV)

In [ ]:
from IPython.display import Audio, display

print("🎙️ RAW AUDIO")
display(Audio(RAW_WAV))

print("\n DEEPFILTERNET3 ENHANCED AUDIO")
display(Audio(ENHANCED_WAV))